# Experiment 2: Equity Index Timing via VRP, Term Structure, VVIX & Trend

This notebook walks through Experiment 2 step-by-step, demonstrating a systematic long/short timing strategy for S&P 500 E-mini futures.

**Strategy overview:** Combine four independent signals to time the market:
1. **VRP** — HAR-derived Variance Risk Premium from Experiment 1
2. **Term Structure Slope** — Daily cross-sectional slope of VIX futures curve
3. **Equity Trend Quotient** — Price / 200-day SMA (above or below trend)
4. **VVIX tail-risk gauge** — 5-day moving average of VVIX (vol-of-VIX)

**Deviations from plan are marked `[DEV-N]` throughout.**

---

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# Paths
ROOT    = Path('.')
DATA    = ROOT.parent / 'data'
VRP_OUT = ROOT.parent / 'vrp_experiment' / 'output'
OUTPUT  = ROOT / 'output'

# Import shared utilities
sys.path.insert(0, str(ROOT.parent / 'bh_replication'))
from har_model import _nw_se

# Import Experiment 2 functions
sys.path.insert(0, str(ROOT))
from experiment2 import (
    load_vrp_series, load_es_front_month, load_vix_futures_term_structure,
    load_vvix, compute_vix_term_slope, compute_trend_quotient, compute_vvix_ma5,
    build_master_panel, run_bivariate_regressions, run_rolling_regression_positions,
    compute_logic_gate_positions, compute_buy_and_hold, compute_ma_trend,
    simulate_strategy, compute_performance_stats
)

%matplotlib inline
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Setup complete.')

---
## Step 1: Load & Inspect Data

We need four data sources:


In [ ]:
# Load all data
vrp   = load_vrp_series()
es    = load_es_front_month()
vx_df = load_vix_futures_term_structure()
vvix  = load_vvix()

print('=== Data Summary ===')
print(f'VRP (from Experiment 1):  {len(vrp):,} obs  '
      f'[{vrp.index.min().date()} – {vrp.index.max().date()}]')
print(f'ES front-month:           {len(es):,} obs  '
      f'[{es.index.min().date()} – {es.index.max().date()}]')
print(f'VIX futures (VX):         {vx_df["date"].nunique():,} dates, '
      f'{len(vx_df):,} contract-days')
print(f'VVIX:                     {len(vvix):,} obs  '
      f'[{vvix.index.min().date()} – {vvix.index.max().date()}]')

In [ ]:
# Preview VRP series from Experiment 1
print('VRP series (VRP = IVar - CV):')
print(vrp.describe().round(3))
print(f'\nVRP > 0: {(vrp["VP"] > 0).mean()*100:.1f}% of days')

In [ ]:
# Preview VIX futures — multiple contracts per day
sample_date = vx_df['date'].iloc[1000]
day_contracts = vx_df[vx_df['date'] == sample_date].sort_values('ttm_years')
print(f'VIX futures on {sample_date.date()}: {len(day_contracts)} contracts')
print(day_contracts[['security','price','ttm_years']].to_string(index=False))

---
## Step 2: VIX Futures Term Structure Slope

For each trading day, fit a cross-sectional linear regression across all listed VIX futures:

$$\text{VIX\_price}_{i,t} = \alpha_t + s_t \cdot \text{TtM}_{i,t} + \epsilon_{i,t}$$

The slope **$s_t$** (VIX points per year) captures the term structure shape:

Historical contango average ≈ 2.32 pts/yr; plan uses **2.0 as the long trigger**.

In [ ]:
# Compute daily VIX term structure slopes
term_slope = compute_vix_term_slope(vx_df)

print(f'Term slope computed for {len(term_slope):,} days')
print(f'Mean slope:            {term_slope.mean():.3f} pts/yr')
print(f'% days contango (>0):  {(term_slope > 0).mean()*100:.1f}%')
print(f'% days backwardation:  {(term_slope < 0).mean()*100:.1f}%')
print(f'% days > 2.0 (long):   {(term_slope > 2.0).mean()*100:.1f}%')
print(f'% days < −1.0 (short): {(term_slope < -1.0).mean()*100:.1f}%')

In [ ]:
# Visualise term structure slope over time
fig, ax = plt.subplots(figsize=(15, 4))
ax.fill_between(term_slope.index, term_slope, 0,
                where=(term_slope >= 0), color='steelblue', alpha=0.5, label='Contango')
ax.fill_between(term_slope.index, term_slope, 0,
                where=(term_slope < 0), color='salmon', alpha=0.6, label='Backwardation')
ax.axhline(2.0,  color='green',     ls='--', lw=1, label='Long threshold (2.0)')
ax.axhline(-1.0, color='firebrick', ls='--', lw=1, label='Short threshold (−1.0)')
ax.axhline(0, color='black', lw=0.5)
ax.set_title('VIX Futures Term Structure Slope (daily cross-sectional OLS)')
ax.set_ylabel('Slope (VIX pts/year)')
ax.legend()
plt.tight_layout()
plt.show()

**Key insight:** The VIX curve spends ~80% of time in contango — consistent with the plan's note that "volatility futures curve resides in contango roughly two-thirds to three-quarters of the time." Backwardation episodes (slope < −1) cluster around crises (2008, 2020, 2022).

---
## Step 3: Equity Trend Quotient (200-day SMA)

$$\text{TrendQuotient}_t = \frac{P_t}{\text{SMA}_{200}(P_t)}$$

> **[DEV-1]** The plan requires the spot S&P 500 index level $P_t$. Since spot data is not directly available, we use the **front-month ES futures price level** (reconstructed as the cumulative product of daily front-month returns, rebased to 1000 at the start). This is a close approximation since ES futures track the index very tightly.

In [ ]:
# Compute trend quotient
trend_q = compute_trend_quotient(es)

print(f'Trend quotient: mean={trend_q.mean():.3f}  min={trend_q.min():.3f}  max={trend_q.max():.3f}')
print(f'% days above SMA200 (bull trend): {(trend_q > 1).mean()*100:.1f}%')

# Plot price level vs SMA200
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
sma200 = es['price_level'].rolling(200).mean()

ax = axes[0]
ax.plot(es.index, es['price_level'], color='steelblue', lw=0.8, label='ES price level')
ax.plot(es.index, sma200, color='darkorange', lw=1.2, label='SMA 200')
ax.fill_between(es.index, es['price_level'], sma200,
                where=(es['price_level'] >= sma200), color='steelblue', alpha=0.15)
ax.fill_between(es.index, es['price_level'], sma200,
                where=(es['price_level'] < sma200), color='salmon', alpha=0.25)
ax.set_title('S&P 500 E-mini Front-Month Price Level vs 200-day SMA')
ax.legend()

ax2 = axes[1]
ax2.plot(trend_q.index, trend_q, color='steelblue', lw=0.7)
ax2.axhline(1.0, color='black', ls='--', lw=0.8)
ax2.set_ylabel('P / SMA200')
ax2.set_title('Trend Quotient (above 1 = bullish)')
plt.tight_layout()
plt.show()

---
## Step 4: VVIX Tail-Risk (5-day MA)

The VVIX measures the **volatility of the VIX** — a gauge of uncertainty and tail-risk hedging demand. High VVIX → elevated fear; low VVIX → calm markets.

> **[DEV-2]** VVIX data starts 2006-03-06, limiting our joint-signal analysis to this date onwards.

Logic gate thresholds:


In [ ]:
# Compute VVIX 5-day SMA
vvix_ma5 = compute_vvix_ma5(vvix)

print(f'VVIX MA5: mean={vvix_ma5.mean():.1f}  std={vvix_ma5.std():.1f}')
print(f'% days < 95  (long-eligible):  {(vvix_ma5 < 95).mean()*100:.1f}%')
print(f'% days > 110 (short-eligible): {(vvix_ma5 > 110).mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(vvix_ma5.index, vvix_ma5, color='purple', lw=0.8, label='VVIX 5-day MA')
ax.fill_between(vvix_ma5.index, vvix_ma5, 95,
                where=(vvix_ma5 > 95), color='salmon', alpha=0.2)
ax.axhline(95,  color='green',     ls='--', lw=1, label='Long threshold (<95)')
ax.axhline(110, color='firebrick', ls='--', lw=1, label='Short threshold (>110)')
ax.set_title('VVIX (Vol-of-VIX) 5-Day Moving Average — Tail-Risk Gauge')
ax.set_ylabel('VVIX MA5')
ax.legend()
plt.tight_layout()
plt.show()

---
## Building the Master Signal Panel

Align all signals and compute the **20-day forward S&P 500 return** (the regression target):

$$R_{t+20} = \left(\prod_{k=1}^{20}(1 + r_{t+k})\right) - 1$$

where $r_{t+k}$ is the daily ES return on day $t+k$. The signal at day $t$ uses only information available at the close of $t$; the return window starts from $t+1$, so there is no look-ahead.


In [ ]:
# Assemble master panel
panel = build_master_panel(vrp, es, term_slope, trend_q, vvix_ma5)

# [DEV-2] Restrict to VVIX overlap period
panel = panel[panel.index >= '2006-03-06'].copy()

print(f'Master panel: {panel.index.min().date()} – {panel.index.max().date()}')
print(f'Observations: {len(panel):,}')
print()
print(panel[['VP', 'term_slope', 'trend_q', 'vvix_ma5', 'fwd_ret_20']].describe().round(4))

In [ ]:
# Four-panel signal overview
fig, axes = plt.subplots(4, 1, figsize=(15, 14), sharex=True)
s, e = panel.index[0], panel.index[-1]

axes[0].fill_between(panel.index, panel['VP'], 0,
    where=(panel['VP'] >= 0), color='steelblue', alpha=0.5, label='VRP > 0')
axes[0].fill_between(panel.index, panel['VP'], 0,
    where=(panel['VP'] < 0), color='salmon', alpha=0.5, label='VRP < 0')
axes[0].axhline(0, color='black', lw=0.5)
axes[0].set_title('Signal 1: Variance Risk Premium (VRP = IVar − CV)')
axes[0].legend(fontsize=8)

axes[1].plot(panel.index, panel['term_slope'], color='darkorange', lw=0.7)
axes[1].axhline(2.0, color='green', ls='--', lw=0.8, label='Long threshold (2.0)')
axes[1].axhline(-1.0, color='firebrick', ls='--', lw=0.8, label='Short threshold (−1.0)')
axes[1].axhline(0, color='black', lw=0.4)
axes[1].set_title('Signal 2: VIX Term Structure Slope')
axes[1].legend(fontsize=8)

axes[2].plot(panel.index, panel['trend_q'], color='steelblue', lw=0.7)
axes[2].axhline(1.0, color='black', ls='--', lw=0.8)
axes[2].set_title('Signal 3: Equity Trend Quotient (P / SMA200)')

axes[3].plot(panel.index, panel['vvix_ma5'], color='purple', lw=0.7)
axes[3].axhline(95, color='green', ls='--', lw=0.8, label='<95 (long-ok)')
axes[3].axhline(110, color='firebrick', ls='--', lw=0.8, label='>110 (short-ok)')
axes[3].set_title('Signal 4: VVIX 5-Day MA')
axes[3].legend(fontsize=8)

axes[-1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.suptitle('All Four Signals — 2006 to Present', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---
## Step 5: Bivariate Predictive Regressions

Regress the 20-day forward S&P 500 return on VRP paired with each signal:

| Model | Formula |
|-------|--------|
| Base  | $R_{t+20} = \alpha + \beta_1 \text{VRP}_t + \epsilon$ |
| A     | $R_{t+20} = \alpha + \beta_1 \text{VRP}_t + \beta_2 s_t + \epsilon$ |
| B     | $R_{t+20} = \alpha + \beta_1 \text{VRP}_t + \beta_2 (P_t/\text{SMA}_{200}) + \epsilon$ |
| C     | $R_{t+20} = \alpha + \beta_1 \text{VRP}_t + \beta_2 \text{VVIX-MA5}_t + \epsilon$ |

Newey-West standard errors with 20 lags to correct for overlapping windows.

In [ ]:
# Run all bivariate predictive regressions
reg_results = run_bivariate_regressions(panel)

# Display results table
print('=== Full-Sample Bivariate Predictive Regressions ===')
print(f'{"Model":<12} {"Adj-R²":>8} {"VP coef":>10} {"VP t-stat":>10} {"Sec. signal":>12} {"Sec. t-stat":>12}')
print('-' * 70)

sec_var = {'Base': None, 'Model_A': 'term_slope', 'Model_B': 'trend_q', 'Model_C': 'vvix_ma5'}
for m in ['Base', 'Model_A', 'Model_B', 'Model_C']:
    r  = reg_results[m]
    vp = r['params']['VP']
    vt = r['t_stat']['VP']
    sv = sec_var[m]
    sc = r['params'].get(sv, np.nan) if sv else np.nan
    st = r['t_stat'].get(sv, np.nan)  if sv else np.nan
    sig_vp = '***' if abs(vt) > 2.58 else ('**' if abs(vt) > 1.96 else ('*' if abs(vt) > 1.645 else ''))
    sig_s  = '***' if abs(st) > 2.58 else ('**' if abs(st) > 1.96 else ('*' if abs(st) > 1.645 else '')) if sv else ''
    print(f'{m:<12} {r["adj_r2"]:>8.4f} {vp:>10.5f} {vt:>8.3f}{sig_vp:<3} '
          f'{sc if not np.isnan(sc) else "—":>12.5f} {st if not np.isnan(st) else "—":>10.3f}{sig_s}')

print('\nSignificance: * p<0.10, ** p<0.05, *** p<0.01 (Newey-West, 20 lags)')

In [ ]:
# Visualise regression results
models = ['Base', 'Model_A', 'Model_B', 'Model_C']
labels = ['Base (VRP)', 'Model A\n(VRP+Slope)', 'Model B\n(VRP+Trend)', 'Model C\n(VRP+VVIX)']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

adj_r2 = [reg_results[m]['adj_r2'] for m in models]
axes[0].bar(labels, adj_r2, color='steelblue', alpha=0.7)
axes[0].set_title('Adj. R²')
axes[0].set_ylabel('Adj. R²')

vp_t = [reg_results[m]['t_stat']['VP'] for m in models]
axes[1].bar(labels, vp_t,
            color=['steelblue' if abs(t) >= 1.96 else 'salmon' for t in vp_t], alpha=0.7)
axes[1].axhline(1.96,  color='grey', ls='--', lw=0.8, label='±1.96')
axes[1].axhline(-1.96, color='grey', ls='--', lw=0.8)
axes[1].set_title('VP t-statistic (NW)')
axes[1].legend(fontsize=8)

sec_key = {'Base': None, 'Model_A': 'term_slope', 'Model_B': 'trend_q', 'Model_C': 'vvix_ma5'}
sec_t = [reg_results[m]['t_stat'].get(sec_key[m], 0) if sec_key[m] else 0 for m in models]
axes[2].bar(labels,
            sec_t,
            color=['steelblue' if abs(t) >= 1.96 else 'salmon' for t in sec_t], alpha=0.7)
axes[2].axhline(1.96,  color='grey', ls='--', lw=0.8, label='±1.96')
axes[2].axhline(-1.96, color='grey', ls='--', lw=0.8)
axes[2].set_title('Secondary Signal t-statistic (NW)')
axes[2].legend(fontsize=8)

plt.suptitle('Bivariate Predictive Regressions — Full Sample, NW SEs (20 lags)', fontsize=12)
plt.tight_layout()
plt.show()

print('\nKey finding:')
print('  Model A (VP + Term Slope): highest Adj-R², both vars significant at 5% — best regression model')
print('  Model C (VP + VVIX):       second-best, VVIX significant at 5%')
print('  Model B (VP + Trend):      trend quotient NOT significant (t=-0.53) — weakest')

---
### Horizon Robustness: VRP → 40-day & VVIX MA5 → 20-day (Expanding Window)

Two supplemental expanding-window analyses testing alternative horizons:

| Regression | OOS gap | NW lags | Method |
|------------|---------|---------|--------|
| $R_{t+40} \sim \text{VRP}_t$ | 40 days | 40 | Expanding window, OOS from 2012-01-01 |
| $R_{t+20} \sim \text{VVIX-MA5}_t$ | 20 days | 20 | Expanding window, OOS from 2012-01-01 |

Same per-day $|t| > 1.28$ gate as all other models. Plots show cumulative OOS return (top) and the NW t-statistic of the predictor over time (bottom).


In [ ]:
from IPython.display import Image, display

print('=== (A) VRP -> 40-day Forward Return (Expanding Window) ===')
display(Image(filename=str(OUTPUT / 'expanding_window' / 'VRP 40-day' / 'symmetric_VRP_40d.png'), width=1200))

print('\n=== (B) VVIX MA5 -> 20-day Forward Return (Expanding Window) ===')
display(Image(filename=str(OUTPUT / 'expanding_window' / 'VVIX MA5' / 'symmetric_VVIX_MA5.png'), width=1200))

---
## Step 6: Rolling Regression Position Generation

Using a 500-day rolling OLS regression, generate out-of-sample predicted 20-day returns $\hat{R}_{t+20}$. Then:

- Long (+1) if $\hat{R}_{t+20} > +\delta$
- Short (−1) if $\hat{R}_{t+20} < -\delta$
- Flat (0) otherwise

We test $\delta \in \{0.5\%, 1.0\%, 2.0\%\}$. The rolling window prevents look-ahead: each day's forecast uses only data available at that point.


In [ ]:
# Generate rolling regression positions for the best model (Model A)
print('Generating Model A (VP + Term Slope) rolling positions...')
pos_A_5   = run_rolling_regression_positions(panel, 'Model_A', 0.005)
pos_A_10  = run_rolling_regression_positions(panel, 'Model_A', 0.010)
pos_A_20  = run_rolling_regression_positions(panel, 'Model_A', 0.020)

for lbl, pos in [('δ=0.5%', pos_A_5), ('δ=1.0%', pos_A_10), ('δ=2.0%', pos_A_20)]:
    print(f'  {lbl}: Long={float((pos==1).mean())*100:.1f}%  '
          f'Short={float((pos==-1).mean())*100:.1f}%  '
          f'Flat={float((pos==0).mean())*100:.1f}%')

---
## Step 7: Algorithmic Logic Gate

The logic gate requires **all four conditions** simultaneously:

| Direction | Conditions |
|-----------|------------|
| **Long** (+1) | VP > rolling_mean + 1σ **AND** term_slope > 2.00 **AND** trend_q > 1.00 **AND** VVIX_MA5 < 95 |
| **Short** (−1) | VP < rolling_mean − 1σ **AND** term_slope < −1.00 **AND** trend_q < 1.00 **AND** VVIX_MA5 > 110 |
| **Flat** (0) | Otherwise |

This very strict conjunctive rule means the strategy is rarely active — consistent with the plan's note on "regime inactivity."

In [ ]:
# Compute logic gate positions
pos_logic = compute_logic_gate_positions(panel)

print('Logic Gate Position Distribution:')
print(f'  Long  (+1): {(pos_logic == 1).sum():4d} days ({(pos_logic==1).mean()*100:.1f}%)')
print(f'  Short (-1): {(pos_logic ==-1).sum():4d} days ({(pos_logic==-1).mean()*100:.1f}%)')
print(f'  Flat  ( 0): {(pos_logic == 0).sum():4d} days ({(pos_logic==0).mean()*100:.1f}%)')
print()
print('Note: strategy is flat ~98% of the time due to strict conjunctive conditions.')
print('This matches the plan\'s "regime inactivity" warning.')

fig, ax = plt.subplots(figsize=(15, 2.5))
ax.fill_between(pos_logic.index, pos_logic, 0,
                where=(pos_logic > 0), color='steelblue', alpha=0.7, label='Long')
ax.fill_between(pos_logic.index, pos_logic, 0,
                where=(pos_logic < 0), color='salmon',    alpha=0.7, label='Short')
ax.set_title('Logic Gate Position History (mostly flat by design)')
ax.set_ylabel('Position')
ax.set_ylim(-1.5, 1.5)
ax.legend()
plt.tight_layout()
plt.show()

---
## Step 8: Strategy Simulation

**Simulation mechanics:**
- Position $p_t \in \{-1, 0, +1\}$ is set at end-of-day $t$ using same-day signals; `pos.shift(1)` applies it to day $t+1$ returns — no look-ahead
- Net daily P&L: $p_{t-1} \times r_t - 0.0005 \times \mathbf{1}[p_t \neq p_{t-1}]$ (0.05% round-trip slippage per trade)
- Cumulative return rebased to 1.0 at first observation

In [ ]:
# Compute baseline positions
daily_ret = panel['daily_ret'].dropna()
es_sub    = es.loc[panel.index[panel.index.isin(es.index)]]

pos_bah   = compute_buy_and_hold(daily_ret)
pos_trend = compute_ma_trend(es_sub).reindex(daily_ret.index).ffill().fillna(0)

# Simulate all strategies
strategies = {
    'Logic Gate':   pos_logic,
    'Buy-and-Hold': pos_bah,
    'MA Trend':     pos_trend,
    'Model_A δ=0.5%': pos_A_5,
    'Model_A δ=1.0%': pos_A_10,
    'Model_A δ=2.0%': pos_A_20,
}

sim_results = {}
for lbl, pos in strategies.items():
    sim = simulate_strategy(pos, daily_ret, label=lbl)
    sim_results[lbl] = sim

# Performance table
print(f'\n{"Strategy":<20} {"Ann.Return":>10} {"Ann.Vol":>8} {"Sharpe":>7} {"MaxDD":>8} {"Total Ret":>10}')
print('-' * 70)
for lbl, sim in sim_results.items():
    s = compute_performance_stats(sim, lbl)
    print(f'{lbl:<20} {s["ann_ret"]*100:>9.2f}% {s["ann_vol"]*100:>7.2f}% '
          f'{s["sharpe"]:>7.3f} {s["max_dd"]*100:>7.2f}% {s["total_ret"]*100:>9.2f}%')

In [ ]:
# Cumulative returns chart
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

colors = {'Logic Gate': 'green', 'Buy-and-Hold': 'steelblue', 'MA Trend': 'darkorange',
          'Model_A δ=0.5%': 'purple', 'Model_A δ=1.0%': 'teal', 'Model_A δ=2.0%': 'brown'}
styles = {'Logic Gate': '-', 'Buy-and-Hold': '--', 'MA Trend': '--',
          'Model_A δ=0.5%': '-', 'Model_A δ=1.0%': '-', 'Model_A δ=2.0%': '-'}

ax = axes[0]
for lbl, sim in sim_results.items():
    cum = sim['cum_net']
    ax.plot(cum.index, cum.values, label=lbl,
            color=colors.get(lbl, 'grey'),
            ls=styles.get(lbl, '-'), lw=1.3, alpha=0.85)
ax.axhline(1, color='black', lw=0.5, ls=':')
ax.set_title('Cumulative Net Returns — All Strategies (2006–2026)')
ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=8)

# Drawdown for main strategies
ax2 = axes[1]
for lbl in ['Logic Gate', 'Buy-and-Hold', 'MA Trend']:
    cum = sim_results[lbl]['cum_net'].dropna()
    dd  = (cum / cum.cummax() - 1)
    ax2.fill_between(dd.index, dd, 0, alpha=0.3, color=colors[lbl], label=lbl)
ax2.set_ylabel('Drawdown')
ax2.set_title('Drawdown Profiles — Main Strategies')
ax2.legend()
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

---
## Step 9: Alpha Isolation and Benchmark Comparison

**Benchmarks:**
1. **Buy-and-Hold** — always long S&P 500 futures
2. **200-day MA Trend** — long when price > SMA200, short otherwise

**Key observations:**
- The logic gate strategy is flat ~98% of the time due to its strict four-factor conjunctive rule — it generates minimal returns but also minimal drawdown
- Rolling OLS Model A (δ=1.0%) achieves a positive Sharpe with more moderate drawdown than buy-and-hold
- MA Trend provides a competitive simple momentum benchmark; it catches most of the 2008 decline by going short

In [ ]:
# Compute Sharpe ratio comparison
perf_table = []
for lbl, sim in sim_results.items():
    s = compute_performance_stats(sim, lbl)
    perf_table.append(s)

perf_df = pd.DataFrame(perf_table).set_index('label')
perf_df[['ann_ret', 'ann_vol', 'sharpe', 'max_dd', 'total_ret']] = (
    perf_df[['ann_ret', 'ann_vol', 'sharpe', 'max_dd', 'total_ret']].round(4)
)
print('Performance Summary:')
print(perf_df[['ann_ret', 'ann_vol', 'sharpe', 'max_dd']].to_string())

In [ ]:
# Sharpe ratio bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

lbls   = list(perf_df.index)
sharpe = perf_df['sharpe'].values
maxdd  = perf_df['max_dd'].values

axes[0].barh(lbls, sharpe,
             color=['steelblue' if s >= 0 else 'salmon' for s in sharpe], alpha=0.7)
axes[0].axvline(0, color='black', lw=0.5)
axes[0].set_title('Sharpe Ratio (0% risk-free rate)')
axes[0].set_xlabel('Sharpe')

axes[1].barh(lbls, maxdd * 100, color='salmon', alpha=0.7)
axes[1].set_title('Maximum Drawdown (%)')
axes[1].set_xlabel('Max DD (%)')

plt.suptitle('Strategy Performance Comparison', fontsize=12)
plt.tight_layout()
plt.show()

---
## Summary of Findings (Rolling OLS, Full Sample)

### Regression Analysis (Step 5)
| Model | Adj R² | VP sig? | Secondary sig? |
|-------|--------|---------|----------------|
| Base (VP only)      | 1.59% | ✓ (t=2.22) | — |
| Model A (VRP+Slope)  | **4.21%** | ✓ (t=3.53) | ✓ slope t=−2.15 |
| Model B (VRP+Trend)  | 1.75% | ✓ (t=2.36) | ✗ trend t=−0.53 |
| Model C (VRP+VVIX)   | 3.50% | ✓ (t=2.61) | ✓ VVIX t=2.35 |

**Model A (VRP + Term Structure Slope)** is the best predictive model. The negative slope coefficient is consistent with the literature: deeper contango → lower subsequent returns (mean reversion). Model B's trend quotient fails the 5% significance threshold.

### Trading Strategy Performance (Steps 6–9, Full Sample)

Results below include the pre-OOS warm-up period (2006–2011) as well as OOS. They are **not pure OOS** — rigorous OOS analysis uses the expanding window in Step 10.

Key observations from the full-sample simulation:
- **Logic gate is too selective** to generate meaningful return (flat ~98% of time)
- **MA Trend** catches the 2008 drawdown and is a strong simple benchmark
- **Model A δ=1.0%** achieves positive Sharpe with reduced drawdown vs buy-and-hold

### Key Deviations from Plan
- **[DEV-1]** Spot S&P 500 → ES front-month proxy (spot unavailable)
- **[DEV-2]** Analysis starts 2006-03-06, not 1990 (VVIX starts 2006)
- **[DEV-3]** Expanding window replaces rolling 500-day OLS in final OOS test (rolling has leverage instability from regime-change windows)

In [ ]:
# Display rolling OLS performance summary (computed live above in cell-28)
print('Rolling OLS Strategy Performance Summary (full sample, including pre-OOS period):')
print()
print(perf_df[['ann_ret', 'ann_vol', 'sharpe', 'max_dd']].to_string())
print()
print('Note: Rigorous OOS results (expanding window, 2012+) are in Step 10 below.')

---
## Step 10: Expanding Window OOS Evaluation

The rolling 500-day OLS has a structural weakness: as the window slides, extreme observations enter and exit. A single crisis (e.g., September 2008) can dominate the 500-day window and flip the OLS slope sign, causing the model to go short when VRP is high — the opposite of the long-run risk premium direction.

**Two remedies were evaluated:**

| Method | Coefficients | Gate | Result |
|--------|-------------|------|--------|
| Expanding window | Re-estimated daily on all history to t−20 | Daily \|t\| > 1.28, all betas jointly | ✓ Adopted |
| Fixed split | Frozen at OOS start | δ-only | Dropped — cannot adapt to any new information |

**Expanding window design:**
- At prediction day i, training data = `sub.iloc[0 : i − 20]` — all history minus the 20-day OOS gap
- The 20-day gap ensures `fwd_ret_20[i−21]` (the last training label, covering days i−20 to i−1) is fully realised before prediction
- OOS starts 2012-01-01; all pre-2012 data is training-only
- Positions and beta series are cached in `output/regression_cache/` for fast reruns

In [ ]:

# Expanding window loop — core logic (illustration only; full run via fixed_split_eval.py)
# At each prediction day i, train OLS on ALL history up to row i-20, then apply NW t-gate.

OOS_START = pd.Timestamp('2012-01-01')
MIN_WIN   = 500
T_THRESH  = 1.28
NW_LAGS   = 20
DELTA     = 0.002   # 0.2%

sub = panel.dropna(subset=['VP', 'fwd_ret_20']).copy()
N = len(sub)
oos_idx = sub.index.searchsorted(OOS_START)
start_i = max(MIN_WIN + 20, oos_idx)

positions_demo = pd.Series(0.0, index=sub.index)

# Run on first 200 OOS rows for illustration
for i in range(start_i, min(start_i + 200, N)):
    train  = sub.iloc[0 : i - 20]
    X_tr   = add_constant(train[['VP']], has_constant='skip')
    res    = OLS(train['fwd_ret_20'], X_tr).fit()
    nw_se  = _nw_se(res, nlags=NW_LAGS)
    t_vp   = res.params['VP'] / nw_se[list(res.params.index).index('VP')]

    if abs(t_vp) <= T_THRESH:
        continue  # gate fails → flat

    X_pred = pd.DataFrame({'const': 1.0, 'VP': [sub['VP'].iloc[i]]})
    y_hat  = res.predict(X_pred.reindex(columns=res.params.index))[0]

    if y_hat > DELTA:
        positions_demo.iloc[i] = 1.0
    elif y_hat < -DELTA:
        positions_demo.iloc[i] = -1.0

oos_demo = positions_demo[positions_demo.index >= OOS_START].iloc[:200]
print('Demo slice (first 200 OOS days):')
print(f'  Long  = {int((oos_demo == 1).sum())}  '
      f'Short = {int((oos_demo == -1).sum())}  '
      f'Flat  = {int((oos_demo == 0).sum())}')
print()
print('Key design points:')
print('  • train = sub.iloc[0 : i-20]  — all history, 20-day OOS gap for non-overlapping labels')
print('  • NW SE with 20 lags to correct for 20-day return overlap')
print('  • Position set to 0 whenever any beta fails |t| > 1.28')
print('  • Full cached results in output/regression_cache/ (12 parquet files)')


In [ ]:

# 10.1  Expanding window OOS plots — one per model
# Each plot: cumulative returns + 4 position panels + t-stat evolution
from IPython.display import Image, display

model_images = [
    (OUTPUT / 'expanding_window' / 'VRP' / 'symmetric_VRP.png',
     'Base (VRP only)'),
    (OUTPUT / 'expanding_window' / 'VRP + Term Slope' / 'symmetric_VRP_+_Term_Slope.png',
     'Model A (VRP + Term Slope)'),
    (OUTPUT / 'expanding_window' / 'VRP + VVIX MA5' / 'symmetric_VRP_+_VVIX_MA5.png',
     'Model C (VRP + VVIX MA5)'),
]

for img_path, label in model_images:
    print(f'\n=== {label} ===')
    display(Image(filename=str(img_path), width=1200))


In [ ]:

# 10.2  Strategy comparison — all expanding-window models (symmetric threshold)
from IPython.display import Image, display

img_path = OUTPUT / 'expanding_window' / 'comparisons' / 'symmetric_comparisons.png'
display(Image(filename=str(img_path), width=1100))
print("symmetric_comparisons.png — cumulative OOS returns for all expanding-window models")
print("  Shows how each model navigated COVID crash, 2022 rate shock, and the 2023-2026 bull run")


In [ ]:

# 10.3  Rolling-mu threshold comparison (alternative threshold design)
from IPython.display import Image, display

img_path = OUTPUT / 'expanding_window' / 'comparisons' / 'base_return_shift_comparisons.png'
display(Image(filename=str(img_path), width=1100))
print("base_return_shift_comparisons.png — rolling-mu threshold variant for all models")
print("  Long: ŷ > μ_500d  |  Short: ŷ < 0")
print("  Both t-stats must exceed ±1.28 gate; shaded region shows below-gate (flat) periods")
print("  Both bivariate models (A, C) cross the gate around the COVID-19 crisis (March–April 2020)")


### 10.4 Interpretation: Why Do Bivariate Models Activate at COVID?

The NW t-statistic for a coefficient $\hat{\beta}_j$ grows approximately as:

$$t_j \approx \frac{\hat{\beta}_j \cdot \sigma_{x_j} \cdot \sqrt{n}}{\sigma_\varepsilon}$$

During **2012–2019** (calm regime), the secondary regressors had very low cross-sectional variation: the VIX term structure oscillated in a narrow contango band (σ ≈ 0.8 pts/yr), and VVIX stayed between 80–100 (σ ≈ 8). Low $\sigma_{x_j}$ → large NW-SE → t below the 1.28 gate, even as n grew.

The **March 2020 crash** introduced high-leverage observations in both regressors simultaneously:
- VIX term structure swung from record steep contango (+8 pts/yr) to deep backwardation (−12 pts/yr) within weeks, then reverted — the extreme observations nearly doubled $\sigma_{x_j}$ for `term_slope`
- VVIX exceeded 200 during peak panic — far beyond the 2012–2019 range, dramatically increasing $\sigma_{x_j}$ for `vvix_ma5`

The spike in regressor variance collapsed the NW standard errors, pushing both t-stats above 1.28 and keeping them there (expanding window retains the COVID observations permanently). This is not data snooping — COVID observations were unseen during 2012–2019 predictions and the coefficients' signs are consistent with prior economics (deeper backwardation → lower forward returns; high VVIX → fear premium that mean-reverts).

The practical implication: **Models A and C should be interpreted as volatility-regime strategies**, not all-weather models. They work when vol markets are in an extreme or transitional regime.


---
## Updated Summary of Findings

### Expanding Window OOS Performance (2012-01-01 onward)

| Strategy | Ann. Return | Sharpe | Max DD | Long% | Flat% | Notes |
|----------|:-----------:|:------:|:------:|:-----:|:-----:|-------|
| **Buy-and-Hold** | +14.16% | 0.856 | −30.2% | 100% | 0% | Benchmark |
| **Base δ=0.2%** | **+16.68%** | **1.024** | −26.4% | 99% | 0% | Best SR overall |
| Base δ=0.5%  | +14.61% | 0.921 | −26.4% | 96% | 4% | |
| Base δ=1.0%  | +4.15%  | 0.407 | −16.6% | 12% | 88% | Over-filtered |
| **Model A δ=0.5%** | +6.21% | **0.597** | −23.1% | 32% | 68% | Activates ~2020 |
| **Model C δ=0.2%** | +6.62% | **0.605** | −26.4% | 41% | 59% | Activates ~2020 |

All stats OOS (2012-01-01 onward). Buy-and-hold OOS SR=0.856 reflects the 2012–2026 bull market, not the full sample including the 2008 GFC.

### Key Takeaways

1. **Base model (VRP only) is the most reliable strategy.** SR=1.024 vs buy-and-hold 0.856, with higher return (+16.68% vs +14.16%) and lower drawdown (−26.4% vs −30.2%). 14-year genuine OOS track record (2012–2026), predominantly long because VRP is positive 66% of days — the model captures the equity risk premium with VRP-driven defensive timing.

2. **Models A and C are regime-conditional.** The joint t-gate keeps them flat for most of 2012–2019 (secondary beta SE too large in calm markets). Their effective OOS track record is ~5 years (2020–2026). Lower absolute returns but meaningful drawdown reduction relative to buy-and-hold.

3. **No short signals materialise in fixed-threshold bivariate models.** The joint gate (VRP t + secondary beta t both above 1.28) is too strict for negative predicted returns across this sample.

4. **Expanding window resolves rolling-window instability.** By permanently accumulating all history, no single crisis epoch dominates the OLS fit, producing stable long-biased positions that track the structural risk premium.

### Deviations from Plan

| Code | Plan | Done | Why |
|------|------|------|-----|
| [DEV-1] | Spot S&P 500 | ES front-month proxy | Spot unavailable |
| [DEV-2] | Full sample from 1990 | From 2006-03-06 | VVIX starts 2006 |
| [DEV-3] | Rolling 500-day OLS | Expanding window | Rolling has leverage instability |
| [DEV-5] | RF threshold optimisation | Not implemented | Out of scope |
